In [1]:
import pandas as pd
from datetime import datetime, timedelta
import requests

In [2]:
# def fetch_exelon(start_date, end_date):
#     """
#     Fetch generation data from Elexon BMRS API for a given date range.
#     Loops through the range in 7-day chunks and returns a combined DataFrame.

#     Args:
#         start_date: Start date in 'YYYY-MM-DD' format
#         end_date:   End date in 'YYYY-MM-DD' format

#     Returns:
#         pd.DataFrame with columns: startTime, settlementPeriod, businessType, psrType, quantity
#     """
#     BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"

#     start = pd.Timestamp(start_date)
#     end = pd.Timestamp(end_date)
#     all_records = []

#     chunk_start = start
#     while chunk_start < end:
#         chunk_end = min(chunk_start + timedelta(days=7), end)

#         params = {
#             "from": chunk_start.strftime("%Y-%m-%d"),
#             "to": chunk_end.strftime("%Y-%m-%d"),
#             "settlementPeriodFrom": 1,
#             "settlementPeriodTo": 50,
#             "format": "json"
#         }

#         response = requests.get(BASE_URL, params=params)
#         response.raise_for_status()
#         json_data = response.json()

#         for period in json_data["data"]:
#             for item in period["data"]:
#                 all_records.append({
#                     "startTime":        period["startTime"],
#                     "settlementPeriod": period["settlementPeriod"],
#                     "businessType":     item["businessType"],
#                     "psrType":          item["psrType"],
#                     "quantity":         item["quantity"]
#                 })

#         chunk_start = chunk_end + timedelta(days=1)

#     df = pd.DataFrame(all_records)
#     df["startTime"] = pd.to_datetime(df["startTime"])
#     df = df.sort_values("startTime").reset_index(drop=True)

#     return df

alter this code so that it only takes in one date as an input and gets 7 days of historical data before this date

In [3]:
# import pandas as pd
# import requests
# from datetime import timedelta

# def fetch_exelon(reference_date):
#     """
#     Fetch generation data from Elexon BMRS API for the 7 days prior to a given date.

#     Args:
#         reference_date: Date in 'YYYY-MM-DD' format

#     Returns:
#         pd.DataFrame with columns:
#         startTime, settlementPeriod, businessType, psrType, quantity
#     """

#     BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"

#     ref = pd.Timestamp(reference_date)
#     start = ref - timedelta(days=7)
#     end = ref - timedelta(days=1)

#     params = {
#         "from": start.strftime("%Y-%m-%d"),
#         "to": end.strftime("%Y-%m-%d"),
#         "settlementPeriodFrom": 1,
#         "settlementPeriodTo": 50,
#         "format": "json"
#     }

#     response = requests.get(BASE_URL, params=params)
#     response.raise_for_status()
#     json_data = response.json()

#     all_records = []

#     for period in json_data["data"]:
#         for item in period["data"]:
#             all_records.append({
#                 "startTime":        period["startTime"],
#                 "settlementPeriod": period["settlementPeriod"],
#                 "businessType":     item["businessType"],
#                 "psrType":          item["psrType"],
#                 "quantity":         item["quantity"]
#             })

#     df = pd.DataFrame(all_records)
#     df["startTime"] = pd.to_datetime(df["startTime"])
#     df = df.sort_values("startTime").reset_index(drop=True)

#     return df

request 336 records of data before a given timedate

In [4]:
import pandas as pd
import requests
from datetime import timedelta

def fetch_exelon(reference_datetime):
    """
    Fetch the 336 settlement-period records (7 days) prior to a given datetime.

    Args:
        reference_datetime: datetime string or pd.Timestamp

    Returns:
        pd.DataFrame with 336 historical records before the reference datetime
    """

    BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"

    ref = pd.Timestamp(reference_datetime)

    # Pull extra day to ensure coverage
    start = ref - timedelta(days=8)
    end = ref

    params = {
        "from": start.strftime("%Y-%m-%d"),
        "to": end.strftime("%Y-%m-%d"),
        "settlementPeriodFrom": 1,
        "settlementPeriodTo": 50,
        "format": "json"
    }

    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    json_data = response.json()

    records = []

    for period in json_data["data"]:
        for item in period["data"]:
            records.append({
                "startTime": period["startTime"],
                "settlementPeriod": period["settlementPeriod"],
                "businessType": item["businessType"],
                "psrType": item["psrType"],
                "quantity": item["quantity"]
            })

    df = pd.DataFrame(records)

    # Convert timestamps
    df["startTime"] = pd.to_datetime(df["startTime"])

    # Settlement period -> exact half-hour timestamp
    df["timestamp"] = df["startTime"] + pd.to_timedelta((df["settlementPeriod"] - 1) * 30, unit="m")

    df = df.sort_values("timestamp")

    # Keep only records before reference datetime
    df = df[df["timestamp"] < ref]

    # Take the last 336 settlement periods
    df = df.tail(336).reset_index(drop=True)

    return df

In [5]:
import requests
import pandas as pd

def get_london_weather_df():
    """
    Fetch 7 days of past weather and 14 days of forecast weather for London
    and return the data as a Pandas DataFrame with 'time' as a column.
    """

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation"
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    hourly = data["hourly"]

    df = pd.DataFrame(hourly)

    # Convert time column to datetime but keep as column
    df["time"] = pd.to_datetime(df["time"])

    return df

In [6]:
weather_df = get_london_weather_df()
weather_df.head()

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,wind_speed_120m,wind_speed_80m,pressure_msl,precipitation
0,2026-03-10 00:00:00,9.6,6.9,100,0.0,0.0,0.0,4.23,4.25,1017.3,0.0
1,2026-03-10 01:00:00,9.3,5.7,93,0.0,0.0,0.0,4.05,4.57,1016.7,0.0
2,2026-03-10 02:00:00,9.1,4.4,94,0.0,0.0,0.0,3.88,5.38,1016.8,0.0
3,2026-03-10 03:00:00,9.4,4.0,93,0.0,0.0,0.0,4.09,4.76,1016.3,0.0
4,2026-03-10 04:00:00,9.3,4.8,96,0.0,0.0,0.0,3.67,4.04,1015.8,0.0


In [7]:
weather_df[:168]

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,wind_speed_120m,wind_speed_80m,pressure_msl,precipitation
0,2026-03-10 00:00:00,9.6,6.9,100,0.0,0.0,0.0,4.23,4.25,1017.3,0.0
1,2026-03-10 01:00:00,9.3,5.7,93,0.0,0.0,0.0,4.05,4.57,1016.7,0.0
2,2026-03-10 02:00:00,9.1,4.4,94,0.0,0.0,0.0,3.88,5.38,1016.8,0.0
3,2026-03-10 03:00:00,9.4,4.0,93,0.0,0.0,0.0,4.09,4.76,1016.3,0.0
4,2026-03-10 04:00:00,9.3,4.8,96,0.0,0.0,0.0,3.67,4.04,1015.8,0.0
...,...,...,...,...,...,...,...,...,...,...,...
163,2026-03-16 19:00:00,11.5,9.3,69,0.0,0.0,0.0,6.67,7.16,1016.0,0.0
164,2026-03-16 20:00:00,11.2,9.1,81,0.0,0.0,0.0,7.34,7.14,1016.4,0.0
165,2026-03-16 21:00:00,10.7,9.4,98,0.0,0.0,0.0,7.04,7.82,1016.6,0.0
166,2026-03-16 22:00:00,10.9,9.4,98,0.0,0.0,0.0,7.54,7.92,1016.7,0.0


In [8]:
import pandas as pd

def convert_to_half_hourly(df):
    """
    Convert hourly weather dataframe to half-hourly intervals
    using time interpolation.
    """

    df = df.copy()

    # ensure datetime
    df["time"] = pd.to_datetime(df["time"])

    # temporarily set index for resampling
    df = df.set_index("time")

    # resample to 30 minute intervals
    df_halfhour = df.resample("30min").interpolate(method="time")

    # restore time as column
    df_halfhour = df_halfhour.reset_index()

    return df_halfhour

In [9]:
df_halfhour = convert_to_half_hourly(weather_df)
df_halfhour[:336]

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,wind_speed_120m,wind_speed_80m,pressure_msl,precipitation
0,2026-03-10 00:00:00,9.60,6.90,100.0,0.0,0.0,0.0,4.230,4.250,1017.30,0.0
1,2026-03-10 00:30:00,9.45,6.30,96.5,0.0,0.0,0.0,4.140,4.410,1017.00,0.0
2,2026-03-10 01:00:00,9.30,5.70,93.0,0.0,0.0,0.0,4.050,4.570,1016.70,0.0
3,2026-03-10 01:30:00,9.20,5.05,93.5,0.0,0.0,0.0,3.965,4.975,1016.75,0.0
4,2026-03-10 02:00:00,9.10,4.40,94.0,0.0,0.0,0.0,3.880,5.380,1016.80,0.0
...,...,...,...,...,...,...,...,...,...,...,...
331,2026-03-16 21:30:00,10.80,9.40,98.0,0.0,0.0,0.0,7.290,7.870,1016.65,0.0
332,2026-03-16 22:00:00,10.90,9.40,98.0,0.0,0.0,0.0,7.540,7.920,1016.70,0.0
333,2026-03-16 22:30:00,10.95,9.50,99.0,0.0,0.0,0.0,7.575,7.710,1016.70,0.0
334,2026-03-16 23:00:00,11.00,9.60,100.0,0.0,0.0,0.0,7.610,7.500,1016.70,0.0


In [10]:
import pandas as pd

def preprocess_weather_data(df, forecast_start_time):
    """
    Convert hourly weather data to half-hourly intervals and
    keep only 336 records before the forecast start time.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Weather dataframe with 'time' column
    forecast_start_time : str or datetime
        First forecast timestamp from API response
    
    Returns
    -------
    pandas.DataFrame
    """

    df = df.copy()

    # ensure datetime
    df["time"] = pd.to_datetime(df["time"])
    forecast_start_time = pd.to_datetime(forecast_start_time)

    # convert to half-hour intervals
    df = df.set_index("time")
    df = df.resample("30min").interpolate(method="time")
    df = df.reset_index()

    # find index of forecast start
    forecast_idx = df[df["time"] >= forecast_start_time].index[0]

    # keep only 336 records before forecast start
    start_idx = max(0, forecast_idx - 336)

    df = df.iloc[start_idx:]

    return df

# Full Weather Forecast Function

In [11]:
import requests
import pandas as pd

def get_processed_london_weather():

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation"
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms",
        "current": "temperature_2m"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    # Create wind_speed_100m
    df["wind_speed_100m"] = (df["wind_speed_120m"] + df["wind_speed_80m"]) / 2

    # Drop original wind speed columns
    df = df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    # Detect forecast start time automatically
    forecast_start = pd.to_datetime(data["current"]["time"])

    # Convert to half-hour intervals
    df = df.set_index("time")
    df = df.resample("30min").interpolate(method="time")
    df = df.reset_index()

    # Find forecast start index
    forecast_idx = df[df["time"] >= forecast_start].index[0]

    # Keep only 336 records before forecast
    start_idx = max(0, forecast_idx - 336)

    df = df.iloc[start_idx:].reset_index(drop=True)

    return df

In [12]:
df = get_processed_london_weather()
df[:336]

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 13:00:00,12.70,7.40,70.0,319.0,175.0,494.0,1014.70,0.0,5.2150
1,2026-03-10 13:30:00,12.95,7.40,85.0,312.0,166.5,478.5,1014.30,0.0,5.7325
2,2026-03-10 14:00:00,13.20,7.40,100.0,305.0,158.0,463.0,1013.90,0.0,6.2500
3,2026-03-10 14:30:00,13.20,8.35,81.5,203.0,149.0,352.0,1013.55,0.0,6.9550
4,2026-03-10 15:00:00,13.20,9.30,63.0,101.0,140.0,241.0,1013.20,0.0,7.6600
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 10:30:00,11.50,9.35,82.5,106.0,225.0,331.0,1015.65,0.0,7.5375
332,2026-03-17 11:00:00,12.00,9.40,70.0,179.0,240.0,419.0,1015.40,0.0,7.3800
333,2026-03-17 11:30:00,12.40,9.35,49.5,291.5,194.0,485.5,1015.25,0.0,7.4350
334,2026-03-17 12:00:00,12.80,9.30,29.0,404.0,148.0,552.0,1015.10,0.0,7.4900


# Get Historical GenMix + Historical and Forecasted Weather

In [42]:
import requests
import pandas as pd


def get_processed_london_weather():
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation"
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms",
        "current": "temperature_2m"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    # Create wind_speed_100m
    df["wind_speed_100m"] = (df["wind_speed_120m"] + df["wind_speed_80m"]) / 2

    # Drop original wind speed columns
    df = df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    # Detect forecast start time automatically
    forecast_start = pd.to_datetime(data["current"]["time"])

    # Convert to half-hour intervals
    df = df.set_index("time")
    df = df.resample("30min").interpolate(method="time")
    df = df.reset_index()

    # Find forecast start index
    forecast_idx = df[df["time"] >= forecast_start].index[0]

    # Keep only 336 records before forecast, plus all forecast rows after
    start_idx = max(0, forecast_idx - 336)
    df = df.iloc[start_idx:].reset_index(drop=True)

    return df, forecast_start

In [43]:
import requests
import pandas as pd


def fetch_elexon_before_forecast_start(forecast_start):
    url = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"

    forecast_start = pd.to_datetime(forecast_start, utc=True).tz_localize(None)
    start_time = forecast_start - pd.Timedelta(days=7)

    params = {
        "from": start_time.strftime("%Y-%m-%dT%H:%MZ"),
        "to": forecast_start.strftime("%Y-%m-%dT%H:%MZ"),
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    json_data = response.json()

    all_records = []
    for period in json_data["data"]:
        for item in period["data"]:
            all_records.append({
                "startTime": period["startTime"],
                "settlementPeriod": period["settlementPeriod"],
                "businessType": item["businessType"],
                "psrType": item["psrType"],
                "quantity": item["quantity"]
            })

    df = pd.DataFrame(all_records)

    if df.empty:
        return df

    df["startTime"] = pd.to_datetime(df["startTime"], utc=True).dt.tz_localize(None)
    df = df.sort_values(["startTime", "settlementPeriod", "psrType"]).reset_index(drop=True)

    return df

In [49]:
def elexon_preproc(df: pd.DataFrame, forecast_start) -> pd.DataFrame:
    df = df.copy()
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True).dt.tz_localize(None)

    df_pivot = df.pivot_table(
        index="startTime",
        columns="psrType",
        values="quantity",
        aggfunc="sum"
    ).reset_index()

    df_pivot.columns.name = None

    fuel_cols = [col for col in df_pivot.columns if col != "startTime"]
    df_pivot["total_output_MW"] = df_pivot[fuel_cols].sum(axis=1)

    # Build exact expected half-hour range: 336 rows
    forecast_start = pd.to_datetime(forecast_start).floor("30min")
    expected_index = pd.date_range(
        start=forecast_start - pd.Timedelta(days=7),
        end=forecast_start - pd.Timedelta(minutes=30),
        freq="30min"
    )

    df_pivot = (
        df_pivot.set_index("startTime")
        .reindex(expected_index)
        .rename_axis("startTime")
        .reset_index()
    )

    return df_pivot

In [50]:
weather_df, forecast_start = get_processed_london_weather()

raw_elexon_df = fetch_elexon_before_forecast_start(forecast_start)
elexon_df = elexon_preproc(raw_elexon_df, forecast_start)

print(elexon_df.shape)

(336, 13)


In [51]:
weather_df[:336]

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 13:30:00,12.95,7.40,85.0,312.0,166.5,478.5,1014.30,0.0,5.7325
1,2026-03-10 14:00:00,13.20,7.40,100.0,305.0,158.0,463.0,1013.90,0.0,6.2500
2,2026-03-10 14:30:00,13.20,8.35,81.5,203.0,149.0,352.0,1013.55,0.0,6.9550
3,2026-03-10 15:00:00,13.20,9.30,63.0,101.0,140.0,241.0,1013.20,0.0,7.6600
4,2026-03-10 15:30:00,13.05,10.25,78.5,58.5,144.0,202.5,1012.95,0.0,7.6400
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 11:00:00,12.00,9.40,70.0,179.0,240.0,419.0,1015.40,0.0,7.3800
332,2026-03-17 11:30:00,12.40,9.35,49.5,291.5,194.0,485.5,1015.25,0.0,7.4350
333,2026-03-17 12:00:00,12.80,9.30,29.0,404.0,148.0,552.0,1015.10,0.0,7.4900
334,2026-03-17 12:30:00,13.30,9.65,15.5,444.5,125.0,569.5,1014.95,0.0,7.2950


In [52]:
elexon_df

,startTime,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-10 13:30:00,2882.0,5201.0,0.0,0.0,1.0,283.0,2853.0,1132.0,5207.0,6171.844,9002.975,32733.819
2,2026-03-10 14:00:00,2881.0,5792.0,0.0,0.0,1.0,284.0,2858.0,475.0,4343.0,6920.148,9274.659,32828.807
3,2026-03-10 14:30:00,2880.0,6201.0,0.0,0.0,1.0,283.0,2853.0,434.0,3904.0,7469.064,9376.497,33401.561
4,2026-03-10 15:00:00,2870.0,6788.0,0.0,0.0,1.0,265.0,2859.0,550.0,3149.0,8138.721,9400.728,34021.449
...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 10:30:00,1404.0,5480.0,0.0,0.0,0.0,459.0,3494.0,1720.0,8446.0,9940.672,8788.734,39732.406
332,2026-03-17 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
333,2026-03-17 11:30:00,1407.0,3875.0,0.0,0.0,0.0,461.0,3504.0,545.0,10774.0,9848.783,8849.988,39264.771
334,2026-03-17 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
import requests
import pandas as pd


def get_processed_london_weather():
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "hourly": [
            "temperature_2m",
            "wind_gusts_10m",
            "cloud_cover",
            "direct_radiation",
            "diffuse_radiation",
            "shortwave_radiation",
            "wind_speed_120m",
            "wind_speed_80m",
            "pressure_msl",
            "precipitation"
        ],
        "timezone": "GMT",
        "forecast_days": 14,
        "past_days": 7,
        "wind_speed_unit": "ms",
        "current": "temperature_2m"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])

    df["wind_speed_100m"] = (df["wind_speed_120m"] + df["wind_speed_80m"]) / 2
    df = df.drop(columns=["wind_speed_120m", "wind_speed_80m"])

    forecast_start = pd.to_datetime(data["current"]["time"])

    df = df.set_index("time")
    df = df.resample("30min").interpolate(method="time")
    df = df.reset_index()

    forecast_idx = df[df["time"] >= forecast_start].index[0]
    start_idx = max(0, forecast_idx - 336)
    df = df.iloc[start_idx:].reset_index(drop=True)

    return df, forecast_start

In [54]:
def fetch_elexon_matching_weather_history(weather_df, forecast_start):
    url = "https://data.elexon.co.uk/bmrs/api/v1/generation/actual/per-type"

    weather_df = weather_df.copy()
    weather_df["time"] = pd.to_datetime(weather_df["time"])

    forecast_start = pd.to_datetime(forecast_start)

    # Use the actual historical part of weather_df
    weather_history = weather_df[weather_df["time"] < forecast_start].copy()

    start_time = weather_history["time"].min()
    end_time = weather_history["time"].max()

    params = {
        "from": start_time.strftime("%Y-%m-%dT%H:%MZ"),
        "to": end_time.strftime("%Y-%m-%dT%H:%MZ"),
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    json_data = response.json()

    all_records = []
    for period in json_data["data"]:
        for item in period["data"]:
            all_records.append({
                "startTime": period["startTime"],
                "settlementPeriod": period["settlementPeriod"],
                "businessType": item["businessType"],
                "psrType": item["psrType"],
                "quantity": item["quantity"]
            })

    df = pd.DataFrame(all_records)

    if df.empty:
        return df

    df["startTime"] = pd.to_datetime(df["startTime"], utc=True).dt.tz_localize(None)

    # Hard clip to exact weather history window
    df = df[
        (df["startTime"] >= start_time) &
        (df["startTime"] <= end_time)
    ].copy()

    df = df.sort_values(["startTime", "settlementPeriod", "psrType"]).reset_index(drop=True)

    return df

In [55]:
def elexon_preproc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True).dt.tz_localize(None)

    df_pivot = df.pivot_table(
        index="startTime",
        columns="psrType",
        values="quantity",
        aggfunc="sum"
    ).reset_index()

    df_pivot.columns.name = None

    fuel_cols = [col for col in df_pivot.columns if col != "startTime"]
    df_pivot["total_output_MW"] = df_pivot[fuel_cols].sum(axis=1)

    return df_pivot.sort_values("startTime").reset_index(drop=True)

In [56]:
weather_df, forecast_start = get_processed_london_weather()

raw_elexon_df = fetch_elexon_matching_weather_history(weather_df, forecast_start)
elexon_df = elexon_preproc(raw_elexon_df)

print("weather start:", weather_df["time"].min())
print("weather hist end:", weather_df[weather_df["time"] < forecast_start]["time"].max())

print("elexon start:", elexon_df["startTime"].min())
print("elexon end:", elexon_df["startTime"].max())

weather start: 2026-03-10 13:30:00
weather hist end: 2026-03-17 13:00:00
elexon start: 2026-03-10 13:30:00
elexon end: 2026-03-17 11:30:00


In [60]:
elexon_df

,startTime,Biomass,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,total_output_MW
0,2026-03-10 13:30:00,2882.0,5201.0,0.0,0.0,1.0,283.0,2853.0,1132.0,5207.0,6171.844,9002.975,32733.819
1,2026-03-10 14:00:00,2881.0,5792.0,0.0,0.0,1.0,284.0,2858.0,475.0,4343.0,6920.148,9274.659,32828.807
2,2026-03-10 14:30:00,2880.0,6201.0,0.0,0.0,1.0,283.0,2853.0,434.0,3904.0,7469.064,9376.497,33401.561
3,2026-03-10 15:00:00,2870.0,6788.0,0.0,0.0,1.0,265.0,2859.0,550.0,3149.0,8138.721,9400.728,34021.449
4,2026-03-10 15:30:00,2877.0,7715.0,0.0,0.0,307.0,269.0,2856.0,581.0,2360.0,8891.152,9143.929,35000.081
...,...,...,...,...,...,...,...,...,...,...,...,...,...
327,2026-03-17 09:00:00,1377.0,5196.0,0.0,0.0,0.0,465.0,3487.0,983.0,4369.0,10212.580,8488.907,34578.487
328,2026-03-17 09:30:00,1398.0,4642.0,0.0,0.0,3.0,453.0,3493.0,820.0,5731.0,10068.318,8526.089,35134.407
329,2026-03-17 10:00:00,1413.0,4652.0,0.0,0.0,0.0,460.0,3486.0,833.0,7046.0,10053.836,8748.913,36692.749
330,2026-03-17 10:30:00,1404.0,5480.0,0.0,0.0,0.0,459.0,3494.0,1720.0,8446.0,9940.672,8788.734,39732.406


In [62]:
weather_df[:336]

,time,temperature_2m,wind_gusts_10m,cloud_cover,direct_radiation,diffuse_radiation,shortwave_radiation,pressure_msl,precipitation,wind_speed_100m
0,2026-03-10 13:30:00,12.95,7.40,85.0,312.0,166.5,478.5,1014.30,0.0,5.7325
1,2026-03-10 14:00:00,13.20,7.40,100.0,305.0,158.0,463.0,1013.90,0.0,6.2500
2,2026-03-10 14:30:00,13.20,8.35,81.5,203.0,149.0,352.0,1013.55,0.0,6.9550
3,2026-03-10 15:00:00,13.20,9.30,63.0,101.0,140.0,241.0,1013.20,0.0,7.6600
4,2026-03-10 15:30:00,13.05,10.25,78.5,58.5,144.0,202.5,1012.95,0.0,7.6400
...,...,...,...,...,...,...,...,...,...,...
331,2026-03-17 11:00:00,12.00,9.40,70.0,179.0,240.0,419.0,1015.40,0.0,7.3800
332,2026-03-17 11:30:00,12.40,9.35,49.5,291.5,194.0,485.5,1015.25,0.0,7.4350
333,2026-03-17 12:00:00,12.80,9.30,29.0,404.0,148.0,552.0,1015.10,0.0,7.4900
334,2026-03-17 12:30:00,13.30,9.65,15.5,444.5,125.0,569.5,1014.95,0.0,7.2950
